# Step 02: Detect and Audit Dataset Anomalies
### Automated 8-Rule Quality Inspector, Multi-Panel Diagnostic Graphs & Visual Sample Audits
Directly integrated with `multi_class_train_rfdetr.ipynb`'s Step B verification (Cell 12).

### 8 Anomaly Detection Rules:
1. **Degenerate Boxes**: Width or height <= 0.
2. **Micro-Clicks**: Accidental clicks with width/height < 4px or area < 16px.
3. **Giant Boxes**: Oversized bounding boxes covering > 95% of image area.
4. **Severe Image Blur**: Blurry images detected via Laplacian variance threshold (< 80.0).
5. **Edge-Cropped / Truncated Boxes**: Bounding boxes abruptly cropped by image borders (touching x=0, y=0, x+w>=W, or y+h>=H).
6. **Exact Duplicates**: Overlapping boxes of the same class with IoU > 0.95.
7. **Loose Margin Padding**: Excessive background padding around tags.
8. **Dense Query Overlaps**: Overly clustered overlapping tags.

### Key Outputs:
- Multi-Panel Anomaly Diagnostic Graphs (Distribution, Aspect Ratios, Areas, Density).
- Visual Sample Inspector displaying flagged anomalies (red boxes & yellow reason tags) vs clean boxes (green).
- `anomalies_manifest.json`: Complete record of all flagged anomaly boxes.
- `cleaned_dataset/`: Sanitized COCO annotations strictly excluding all anomalies.

In [ ]:
# STEP 0 - Environment Dependencies
%pip install -q pycocotools Pillow matplotlib pandas numpy tabulate tqdm

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
print_vram_usage("Init")
logger.info("=" * 65)


import io
def show_plt_figure(fig=None):
    """Guarantees inline plot display in Jupyter under headless and Agg backends."""
    if fig is None:
        fig = plt.gcf()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=100)
    buf.seek(0)
    pil_img = Image.open(buf)
    plt.close(fig)
    display(pil_img)

In [ ]:
# CELL 2 - Paths, Output Setup & Field Configuration
REPO_ROOT = next((p.resolve() for p in [Path(".."), Path("."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path("..").resolve())
INPUT_JSON_DIR = REPO_ROOT / "coco_files"

# Auto-discover active images pool directory across standard pipeline locations
candidate_image_dirs = [
    REPO_ROOT / "multi_class_train_rfdetr" / "images",
    INPUT_JSON_DIR / "images",
    REPO_ROOT / "images",
    Path("images")
]
IMAGES_DIR = next((p for p in candidate_image_dirs if p.exists() and any(p.glob("*"))), candidate_image_dirs[0])
os.makedirs(IMAGES_DIR, exist_ok=True)

# COCO annotation field mappings
IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"

# ==============================================================================
# MASTER ANOMALY AUDIT & SANITIZATION TOGGLES
# ==============================================================================
FILTER_EDGE_CROPPED_BOXES = True   # Automatically remove cut-off/truncated border tags
FILTER_MICRO_DEGENERATE = True     # Automatically remove accidental micro-clicks (<4px)

OUTPUT_ANOMALIES_DIR = Path("anomalies_audit")
OUTPUT_ANOMALIES_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DATASET_DIR = Path("cleaned_dataset")
CLEANED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Input JSON Dir:        {INPUT_JSON_DIR.resolve()}")
logger.info(f"Active Images Pool:    {IMAGES_DIR.resolve()} ({len(list(IMAGES_DIR.glob('*'))):,} cached images)")
logger.info(f"Anomalies Output Dir:  {OUTPUT_ANOMALIES_DIR.resolve()}")
logger.info(f"Cleaned Dataset Dir:   {CLEANED_DATASET_DIR.resolve()}")


In [ ]:
# CELL 4 - Read all annotation JSON files from coco_files/
logger.info("=" * 70)
logger.info(f"[CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
logger.info("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
logger.info(f"   - Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
for jf in tqdm(json_files, desc="[CELL 4] Reading JSON files", unit="file"):
    logger.info(f"   - Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, list):
                raw_records.extend(parsed)
            elif isinstance(parsed, dict):
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

logger.info(f"Total raw annotation records loaded: {len(raw_records)} from {len(json_files)} file(s).")

In [ ]:
# CELL 5 - Parse Annotations & Auto-Discover Multiple Categories
logger.info("=" * 70)
logger.info("[CELL 5] Parsing annotations and discovering categories dynamically...")
logger.info("=" * 70)

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}

for item in tqdm(raw_records, desc="[CELL 5] Parsing annotations", unit="rec"):
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    raw_cat = item.get(CATEGORY_FIELD, ["default"])
    if isinstance(raw_cat, list):
        cat_name = str(raw_cat[0]) if len(raw_cat) > 0 else "default"
    else:
        cat_name = str(raw_cat)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
        if w <= 0 or h <= 0:
            continue
    except (ValueError, TypeError):
        continue

    ann_dict = {
        "bbox": [x, y, w, h],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", [])
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

AUTO_CATEGORIES = sorted(list(category_counts.keys()))
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

logger.info(f"   - Unique images found:         {len(image_annotations)}")
logger.info(f"   - Total valid bounding boxes:  {total_boxes}")
logger.info(f"   - Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    logger.info(f"       - ID {cat_to_id[cat]}: '{cat}' ({count} boxes, {count/total_boxes*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes}")
# Construct definitions for downstream sanitized dataset exports
categories_list = [{"id": idx, "name": cat, "supercategory": "none"} for cat, idx in cat_to_id.items()]
annotations_by_image = image_annotations
annotations_by_cat: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
for anns in image_annotations.values():
    for a in anns:
        annotations_by_cat[a["category_name"]].append(a)

In [ ]:
# CELL 5 - 8-Rule Automated High-Performance Anomaly Detector Engine
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

def compute_box_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    interW = max(0.0, xB - xA)
    interH = max(0.0, yB - yA)
    interArea = interW * interH
    areaA = boxA[2] * boxA[3]
    areaB = boxB[2] * boxB[3]
    denom = areaA + areaB - interArea
    return interArea / denom if denom > 0 else 0.0

# Image path resolver for fast file lookups
search_dirs = [
    str(IMAGES_DIR) if "IMAGES_DIR" in globals() else "images",
    str(REPO_ROOT / "multi_class_train_rfdetr" / "images") if "REPO_ROOT" in globals() else None,
    str(INPUT_JSON_DIR / "images") if "INPUT_JSON_DIR" in globals() else None,
    str(Path("coco_files/images")),
    str(Path("images")),
    "/home/jupyter/multi_class_train_rfdetr/images",
    "/home/jupyter/coco_files/images",
    "/home/jupyter/images"
]
search_dirs = [d for d in search_dirs if d and os.path.exists(d)]

def resolve_image_path(img_identifier):
    img_str = str(img_identifier)
    fname = Path(img_str).name
    stem = Path(img_str).stem
    candidates = [fname, f"{stem}.jpg", f"{img_str}.jpg", img_str]
    for d in search_dirs:
        for c in candidates:
            p = os.path.join(d, c)
            if os.path.exists(p) and os.path.getsize(p) > 0:
                return p
    return None

def compute_single_image_blur(img_identifier) -> Tuple[str, float]:
    """Computes Laplacian variance sharpness score (lower = more blurry)."""
    local_p = resolve_image_path(img_identifier)
    if not local_p:
        return img_identifier, 999.0
    try:
        try:
            import cv2
            gray = cv2.imread(local_p, cv2.IMREAD_GRAYSCALE)
            if gray is not None:
                # Downsample large images for 10x speedup
                if gray.shape[1] > 640:
                    gray = cv2.resize(gray, (640, int(gray.shape[0] * 640 / gray.shape[1])), interpolation=cv2.INTER_AREA)
                return img_identifier, float(cv2.Laplacian(gray, cv2.CV_64F).var())
        except Exception:
            pass

        im = Image.open(local_p).convert("L")
        if im.width > 640:
            scale = 640 / im.width
            im = im.resize((640, int(im.height * scale)), Image.BILINEAR)
        arr = np.array(im, dtype=np.float64)
        lap = (
            -4.0 * arr[1:-1, 1:-1]
            + arr[:-2, 1:-1]
            + arr[2:, 1:-1]
            + arr[1:-1, :-2]
            + arr[1:-1, 2:]
        )
        return img_identifier, float(lap.var())
    except Exception:
        return img_identifier, 999.0

# 1. Fast indexed lookup of raw image dimensions
raw_rec_lookup = {}
for r in raw_records:
    if isinstance(r, dict):
        cu = r.get("coco_url")
        u = r.get("url")
        if cu: raw_rec_lookup[cu] = r
        if u: raw_rec_lookup[u] = r

# 2. Precompute blur scores across images concurrently with multi-threading and tqdm
image_source = annotations_by_image if 'annotations_by_image' in globals() else image_annotations
unique_images = list(image_source.keys())

_image_blur_scores: Dict[str, float] = {}
BLUR_THRESHOLD = 80.0

logger.info(f"[CELL 5] Parallel blur analysis across {len(unique_images)} images using ThreadPoolExecutor...")
with ThreadPoolExecutor(max_workers=min(16, (os.cpu_count() or 4) * 2)) as executor:
    futures = {executor.submit(compute_single_image_blur, iid): iid for iid in unique_images}
    for fut in tqdm(as_completed(futures), total=len(unique_images), desc="[CELL 5] Blur & Sharpness Scan", unit="img"):
        try:
            iid, score = fut.result()
            _image_blur_scores[iid] = score
        except Exception:
            pass

flagged_anomalies = []
clean_annotations = []

# 3. Fast audit loop with tqdm progress bar
logger.info(f"[CELL 5] Executing 8-rule anomaly audit across {len(image_source)} image sets...")
for img_url, anns in tqdm(image_source.items(), desc="[CELL 5] Anomaly Detection Audit", unit="img"):
    img_record = raw_rec_lookup.get(img_url, {})
    img_w = float(img_record.get("width", 1920) or 1920)
    img_h = float(img_record.get("height", 1080) or 1080)
    
    blur_score = _image_blur_scores.get(img_url, 999.0)
    is_image_blurry = blur_score < BLUR_THRESHOLD
    
    num_anns = len(anns)
    for i, a in enumerate(anns):
        bbox = a.get("bbox", [])
        if len(bbox) != 4:
            flagged_anomalies.append({
                "annotation": a,
                "image_url": img_url,
                "reasons": ["Malformed bbox dimensions"],
                "primary_rule": "Malformed bbox dimensions",
                "bbox": [0,0,0,0],
                "category_name": a.get("category_name", "Unknown")
            })
            continue
            
        x, y, w, h = bbox[0], bbox[1], bbox[2], bbox[3]
        area = w * h
        is_anomaly = False
        reasons = []
        
        # Rule 1: Degenerate boxes
        if w <= 0 or h <= 0:
            is_anomaly = True
            reasons.append(f"Degenerate box (w={w}, h={h})")
            
        # Rule 2: Micro-clicks
        if w < 4 or h < 4 or area < 16:
            is_anomaly = True
            reasons.append(f"Micro-click (w={w:.1f}, h={h:.1f}, area={area:.1f})")
            
        # Rule 3: Giant boxes
        if w > 0.95 * img_w and h > 0.95 * img_h:
            is_anomaly = True
            reasons.append(f"Giant box covering >95% image (w={w:.1f}, h={h:.1f})")
            
        # Rule 4: Severe image blur
        if is_image_blurry:
            is_anomaly = True
            reasons.append(f"Blurry image (Laplacian var={blur_score:.1f} < {BLUR_THRESHOLD})")
            
        # Rule 5: Edge-Cropped / Truncated boxes (touching/clipped by image boundaries)
        # Note: Out-of-bounds coordinates are no longer treated as anomalies; cropped boxes at borders are flagged.
        is_cropped_left = x <= 1.0
        is_cropped_top = y <= 1.0
        is_cropped_right = (x + w) >= (img_w - 1.0)
        is_cropped_bottom = (y + h) >= (img_h - 1.0)
        
        if is_cropped_left or is_cropped_top or is_cropped_right or is_cropped_bottom:
            border_sides = []
            if is_cropped_left: border_sides.append("left")
            if is_cropped_top: border_sides.append("top")
            if is_cropped_right: border_sides.append("right")
            if is_cropped_bottom: border_sides.append("bottom")
            is_anomaly = True
            reasons.append(f"Cropped box at image border ({', '.join(border_sides)})")
            
        # Fast overlap checks only if image has multiple boxes
        if num_anns > 1:
            # Rule 6: Duplicate overlap
            for j in range(num_anns):
                if i != j and a.get("category_name") == anns[j].get("category_name"):
                    b_box = anns[j].get("bbox", [])
                    # Quick bounding box distance check before exact IoU
                    if len(b_box) == 4 and abs(x - b_box[0]) < 15 and abs(y - b_box[1]) < 15:
                        iou = compute_box_iou(bbox, b_box)
                        if iou > 0.95:
                            is_anomaly = True
                            reasons.append(f"Duplicate box with IoU {iou:.2f}")
                            break
                        
            # Rule 8: Dense cluster overlaps
            overlapping_count = 0
            for j in range(num_anns):
                if i != j:
                    b_box = anns[j].get("bbox", [])
                    if len(b_box) == 4:
                        if compute_box_iou(bbox, b_box) > 0.70:
                            overlapping_count += 1
                            if overlapping_count >= 3:
                                break
            if overlapping_count >= 3:
                is_anomaly = True
                reasons.append(f"Dense cluster overlap ({overlapping_count} overlapping boxes)")
                    
        # Rule 7: Loose shelf padding
        if w > 0.65 * img_w and h < 0.08 * img_h:
            is_anomaly = True
            reasons.append("Loose shelf padding across multiple bays")
            
        if is_anomaly:
            primary = reasons[0].split("(")[0].strip() if reasons else "Other anomaly"
            flagged_anomalies.append({
                "annotation": a,
                "image_url": img_url,
                "reasons": reasons,
                "primary_rule": primary,
                "bbox": bbox,
                "category_name": a.get("category_name", "Unknown")
            })
        else:
            clean_annotations.append(a)

logger.info("=" * 70)
logger.info(f"ANOMALY AUDIT COMPLETE: {len(flagged_anomalies)} / {total_boxes} boxes flagged ({len(flagged_anomalies)/max(1, total_boxes)*100:.2f}%)")
logger.info(f"Clean annotations retained: {len(clean_annotations)}")
logger.info("=" * 70)

In [ ]:
# CELL 6 - Comprehensive Anomaly Diagnostic Graphs
rule_counts = Counter()
for item in flagged_anomalies:
    for r in item["reasons"]:
        rule_counts[r.split("(")[0].strip()] += 1

fig, axs = plt.subplots(2, 2, figsize=(16, 10))

# Panel 1: Anomaly Rule Frequency Breakdown
labels = list(rule_counts.keys()) or ["No Anomalies"]
counts = list(rule_counts.values()) or [0]
bars = axs[0, 0].barh(labels, counts, color="#E53935")
axs[0, 0].set_title("Flagged Anomalies by Rule Category", fontsize=12, fontweight="bold")
axs[0, 0].set_xlabel("Count")
for b in bars:
    axs[0, 0].text(b.get_width() + 0.5, b.get_y() + b.get_height()/2.0, str(int(b.get_width())), va="center", fontsize=10)

# Panel 2: Image Blur Score Distribution (Sharpness vs Blurry)
blur_scores_list = [s for s in _image_blur_scores.values() if s < 900.0] if "_image_blur_scores" in globals() else []
if blur_scores_list:
    axs[0, 1].hist(blur_scores_list, bins=25, color="#00897B", alpha=0.75, label="Images Inspected")
    axs[0, 1].axvline(BLUR_THRESHOLD, color="red", linestyle="--", linewidth=2, label=f"Blur Threshold (<{BLUR_THRESHOLD})")
    axs[0, 1].set_title("Image Sharpness (Laplacian Variance Distribution)", fontsize=12, fontweight="bold")
    axs[0, 1].set_xlabel("Laplacian Variance (Sharpness Score)")
    axs[0, 1].legend()
else:
    axs[0, 1].text(0.5, 0.5, "Blur audit skipped (No local images evaluated)", ha="center", va="center", fontsize=11)
    axs[0, 1].set_title("Image Sharpness Audit", fontsize=12, fontweight="bold")

# Panel 3: Box Area (Log-scale) Comparison
clean_areas = [a["bbox"][2] * a["bbox"][3] for a in clean_annotations if len(a.get("bbox", [])) == 4 and a["bbox"][2]*a["bbox"][3] > 0]
anom_areas = [a["bbox"][2] * a["bbox"][3] for a in flagged_anomalies if len(a.get("bbox", [])) == 4 and a["bbox"][2]*a["bbox"][3] > 0]

axs[1, 0].hist(np.log10(clean_areas), bins=30, alpha=0.7, label="Clean", color="#1E88E5")
if anom_areas:
    axs[1, 0].hist(np.log10(anom_areas), bins=30, alpha=0.7, label="Anomalies", color="#E53935")
axs[1, 0].set_title("Bounding Box Area Distribution (log10 px²)", fontsize=12, fontweight="bold")
axs[1, 0].set_xlabel("log10(Area)")
axs[1, 0].legend()

# Panel 4: Quality Retention Overview Pie Chart
clean_count = len(clean_annotations)
flagged_count = len(flagged_anomalies)
if total_boxes > 0:
    axs[1, 1].pie([clean_count, flagged_count], labels=[f"Clean ({clean_count})", f"Flagged ({flagged_count})"], autopct="%1.1f%%", colors=["#43A047", "#E53935"], startangle=140, explode=(0, 0.1))
    axs[1, 1].set_title("Dataset Quality Split (Retained vs Excluded)", fontsize=12, fontweight="bold")

plt.tight_layout()
show_plt_figure(fig)

In [ ]:
# CELL 7 - Visual Display of Detected Anomaly Samples (Distinct Samples per Anomaly Type)
import os, sys, json
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from collections import defaultdict, Counter
from pathlib import Path

def display_anomaly_breakdown_and_samples(anomalies_list, images_dir: Path, samples_per_anomaly: int = 5, max_display_width: int = 700):
    """
    1. Computes and displays clean counts for each flagged anomaly rule.
    2. Groups anomalies by distinct rule type.
    3. Displays up to samples_per_anomaly (default 5) visual image previews for EACH anomaly type,
       ensuring only the TARGET anomaly for that specific category is highlighted in red, with
       priority given to images unique to that anomaly category across rules.
    """
    print('=' * 80)
    print('[CELL 7] ANOMALY DETECTION BREAKDOWN & SAMPLE INSPECTION')
    print('=' * 80)

    if not anomalies_list:
        print('Zero anomalies detected! Dataset bounding boxes passed all quality audit checks.')
        return

    # Count frequencies of each anomaly rule category
    rule_counts = Counter()
    anomalies_by_rule = defaultdict(list)
    for a in anomalies_list:
        rules = [r.split('(')[0].strip() for r in a.get('reasons', ['Unspecified anomaly'])]
        for r in set(rules):
            rule_counts[r] += 1
            anomalies_by_rule[r].append(a)

    total_flagged = len(anomalies_list)
    print(f'Total Flagged Anomaly Boxes: {total_flagged}')
    print(f'Distinct Anomaly Types Detected: {len(rule_counts)}')
    print()
    print('COUNTS OF EACH ANOMALY TYPE:')
    print('-' * 60)
    print(f'{"Anomaly Rule Category":<42} | {"Count":<8} | Share')
    print('-' * 60)
    for rule, count in rule_counts.most_common():
        pct = (count / max(1, total_flagged)) * 100
        print(f'{rule:<42} | {count:<8} | {pct:5.1f}%')
    print('-' * 60)
    print()

    # Search paths for resolving images on disk
    search_dirs = [
        str(images_dir),
        str(REPO_ROOT / 'multi_class_train_rfdetr' / 'images') if 'REPO_ROOT' in globals() else None,
        str(INPUT_JSON_DIR / 'images') if 'INPUT_JSON_DIR' in globals() else None,
        str(Path('coco_files/images')),
        str(Path('images')),
        '/home/jupyter/multi_class_train_rfdetr/images',
        '/home/jupyter/coco_files/images',
        '/home/jupyter/images'
    ]
    search_dirs = [d for d in search_dirs if d and os.path.exists(d)]

    def resolve_image_path(img_identifier):
        img_str = str(img_identifier)
        fname = Path(img_str).name
        stem = Path(img_str).stem
        candidates = [fname, f'{stem}.jpg', f'{img_str}.jpg', img_str]
        for d in search_dirs:
            for c in candidates:
                p = os.path.join(d, c)
                if os.path.exists(p) and os.path.getsize(p) > 0:
                    return p
        return None

    img_ann_lookup = annotations_by_image if 'annotations_by_image' in globals() else image_annotations
    globally_displayed_images = set()

    # Render up to samples_per_anomaly for each detected anomaly rule
    for rule, items in tqdm(rule_counts.most_common(), desc="[CELL 7] Inspecting Anomaly Types", unit="rule"):
        rule_items = anomalies_by_rule[rule]
        print('=' * 80)
        print(f'>>> ANOMALY CATEGORY: [{rule.upper()}] (Total: {len(rule_items)})')
        print('=' * 80)

        # Sort candidate images: prefer fresh images not yet displayed in other anomaly sections
        fresh_items = [a for a in rule_items if a.get('image_url') not in globally_displayed_images]
        fallback_items = [a for a in rule_items if a.get('image_url') in globally_displayed_images]
        ordered_candidates = fresh_items + fallback_items

        seen_in_this_rule = set()
        rendered_count = 0

        for anom in ordered_candidates:
            if rendered_count >= samples_per_anomaly:
                break
            img_url = anom.get('image_url')
            if img_url in seen_in_this_rule:
                continue

            local_path = resolve_image_path(img_url)
            if not local_path:
                continue

            seen_in_this_rule.add(img_url)
            globally_displayed_images.add(img_url)
            fname = Path(local_path).name

            try:
                img = Image.open(local_path).convert('RGB')
            except Exception as e:
                print(f'[WARN] Failed opening {local_path}: {e}')
                continue

            im_w, im_h = img.size
            draw = ImageDraw.Draw(img)

            # Filter ONLY the anomalies in this image that match the current rule
            matching_anoms_in_img = [
                a for a in anomalies_list 
                if a.get('image_url') == img_url and any(r.split('(')[0].strip() == rule for r in a.get('reasons', []))
            ]
            matching_boxes = [a.get('bbox') for a in matching_anoms_in_img]

            # 1. Draw valid/clean boxes (and boxes belonging to other categories) in green
            all_boxes_in_img = img_ann_lookup.get(img_url, [])
            for c in all_boxes_in_img:
                cb = c.get('bbox', [])
                if len(cb) == 4 and cb not in matching_boxes:
                    x0 = max(0.0, min(float(im_w - 1), float(cb[0])))
                    y0 = max(0.0, min(float(im_h - 1), float(cb[1])))
                    x1 = max(0.0, min(float(im_w - 1), float(cb[0] + cb[2])))
                    y1 = max(0.0, min(float(im_h - 1), float(cb[1] + cb[3])))
                    draw.rectangle([x0, y0, x1, y1], outline=(0, 230, 118), width=2)

            # 2. Draw ONLY the target rule anomaly boxes in red with yellow callout badge
            for a_item in matching_anoms_in_img:
                ab = a_item.get('bbox', [0, 0, 0, 0])
                if len(ab) != 4:
                    continue
                ax0 = max(0.0, min(float(im_w - 1), float(ab[0])))
                ay0 = max(0.0, min(float(im_h - 1), float(ab[1])))
                ax1 = max(0.0, min(float(im_w - 1), float(ab[0] + max(1.0, ab[2]))))
                ay1 = max(0.0, min(float(im_h - 1), float(ab[1] + max(1.0, ab[3]))))
                
                # Red highlight rectangle
                draw.rectangle([ax0, ay0, ax1, ay1], outline=(229, 57, 53), width=4)

                # Yellow reason badge
                badge_text = f" ! {a_item.get('category_name', 'Box')}: " + ' | '.join(a_item.get('reasons', [])) + ' '
                if hasattr(draw, 'textbbox'):
                    tb = draw.textbbox((ax0, max(0.0, ay0 - 18.0)), badge_text)
                    draw.rectangle(tb, fill=(255, 214, 0))
                    draw.text((ax0, max(0.0, ay0 - 18.0)), badge_text, fill=(0, 0, 0))
                else:
                    draw.text((ax0 + 2, max(0.0, ay0 - 14.0)), badge_text, fill=(229, 57, 53))

            rendered_count += 1
            print(f'Sample {rendered_count}/{samples_per_anomaly} for [{rule}] | Image: {fname} ({im_w}x{im_h}) | Matching Anomaly Boxes: {len(matching_anoms_in_img)}')
            print(f'Flagged Reason: {" | ".join(anom.get("reasons", []))}')

            # Scale for display
            if im_w > max_display_width:
                scale = max_display_width / im_w
                display_img = img.resize((max_display_width, int(im_h * scale)), Image.BILINEAR)
            else:
                display_img = img

            display(display_img)

        if rendered_count == 0:
            print(f'[INFO] Image files corresponding to {rule} occurrences were not found locally on disk.')

    print('=' * 80)
    print('Anomaly sample visualization complete.')
    print('=' * 80)

# Execute the 5-sample per anomaly inspector
display_anomaly_breakdown_and_samples(flagged_anomalies, IMAGES_DIR, samples_per_anomaly=5)

In [ ]:
# CELL 8 - Export Manifest & Sanitized Dataset
manifest_path = OUTPUT_ANOMALIES_DIR / "anomalies_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump({
        "flagged_count": len(flagged_anomalies),
        "total_audited": total_boxes,
        "clean_count": len(clean_annotations),
        "anomalies": flagged_anomalies
    }, f, indent=2)

logger.info(f"✓ Saved anomalies manifest: {manifest_path}")

clean_coco = {
    "images": list({a.get("image_id"): {"id": a.get("image_id"), "file_name": Path(a.get("image_url", "")).name or f"{a.get('image_id')}.jpg", "coco_url": a.get("image_url")} for a in clean_annotations}.values()),
    "annotations": clean_annotations,
    "categories": categories_list
}
clean_json_path = CLEANED_DATASET_DIR / "_annotations.coco.json"
with open(clean_json_path, "w", encoding="utf-8") as f:
    json.dump(clean_coco, f, indent=2)

logger.info(f"✓ Saved sanitized dataset: {clean_json_path} ({len(clean_coco['annotations'])} clean annotations)")

In [ ]:
# CELL 9 - Step B Verification on Cleaned Dataset
def load_coco_info(annotation_path: str) -> dict:
    """Parse a COCO annotation file and return summary statistics."""
    with open(annotation_path) as f:
        data = json.load(f)
    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    num_images = len(data.get("images", []))
    num_anns = len(data.get("annotations", []))
    ann_per_cat: dict = {}
    for ann in data.get("annotations", []):
        cat_name = categories.get(ann["category_id"], "unknown")
        ann_per_cat[cat_name] = ann_per_cat.get(cat_name, 0) + 1
    return {
        "num_images": num_images,
        "num_anns": num_anns,
        "num_classes": len(categories),
        "categories": categories,
        "ann_per_cat": ann_per_cat,
        "raw": data,
    }

# Verify sanitized output file exists
target_cleaned_ann = clean_json_path if 'clean_json_path' in globals() else CLEANED_DATASET_DIR / "_annotations.coco.json"
if target_cleaned_ann.exists():
    cleaned_info = load_coco_info(str(target_cleaned_ann))
    logger.info("=" * 65)
    logger.info(f"Cleaned Dataset Audit: {cleaned_info['num_images']} images, {cleaned_info['num_anns']} annotations, {cleaned_info['num_classes']} classes")
    logger.info(f"Classes: {cleaned_info['categories']}")
    logger.info("=" * 65)
else:
    logger.info(f"Cleaned dataset file {target_cleaned_ann} not generated yet.")


In [ ]:
import json, os, sys
from pathlib import Path
# CELL 10 - Anomaly Selection & Pipeline Export Configuration (Gate to Step 03)
# ==============================================================================
# PIPELINE CONTROL TOGGLE:
# EXCLUDE_ANOMALIES_FROM_TRAINING:
#   - True  (Recommended): Exclude all selected anomalies from downstream training.
#                          Sanitizes the training dataset and exports cleaned annotations.
#   - False (Audit Only): Retain all annotations for training while logging anomalies.
#
# ACTIVE ANOMALY RULES CONFIGURATION:
# Set True / False to include or exclude specific anomaly types from being filtered:
# ==============================================================================

# ==============================================================================
# MASTER ANOMALY SELECTION TOGGLES
# ==============================================================================
EXCLUDE_ANOMALIES_FROM_TRAINING = True
FILTER_EDGE_CROPPED_BOXES = globals().get('FILTER_EDGE_CROPPED_BOXES', True)
FILTER_MICRO_DEGENERATE = globals().get('FILTER_MICRO_DEGENERATE', True)

CONSIDER_ANOMALIES = {
    'Cropped box at image border': FILTER_EDGE_CROPPED_BOXES,   # Automatically remove cut-off/truncated border tags
    'Micro-click': FILTER_MICRO_DEGENERATE,                     # Automatically remove accidental micro-clicks (<4px)
    'Degenerate box': FILTER_MICRO_DEGENERATE,                  # Width or height <= 0
    'Giant box covering >95% image': True,                      # Full-frame overcovering boxes
    'Blurry image': True,                                       # Blurry images (Laplacian variance < 80.0)
    'Duplicate box with IoU': True,                             # Duplicate boxes of same class with IoU > 0.95
    'Dense cluster overlap': True,                              # Excessive cluster overlap with >= 3 boxes
    'Loose shelf padding': False,                               # Optional loose padding rule
    'Malformed bbox dimensions': True                           # Incomplete or non-4-element bounding boxes
}

print('=' * 80)
print('[CELL 10] ANOMALY SELECTION & PIPELINE EXPORT CONFIGURATION')
print('=' * 80)
print(f'Mode: {"EXCLUDE ANOMALIES FROM TRAINING" if EXCLUDE_ANOMALIES_FROM_TRAINING else "AUDIT ONLY (Retain all annotations)"}')
print()
print('Active Anomaly Filters:')
for rule_name, active in CONSIDER_ANOMALIES.items():
    status_str = 'EXCLUDE from training' if active else 'KEEP in training'
    mark = 'X' if active else ' '
    print(f'  [{mark}] {rule_name:<35} -> {status_str}')
print('-' * 80)

# Filter anomalies according to CONSIDER_ANOMALIES configuration
final_excluded_anomalies = []
final_retained_annotations = []

for anom in flagged_anomalies:
    # Check if this anomaly matches any active rule
    matched_active_rule = False
    for r in anom.get('reasons', []):
        r_key = r.split('(')[0].strip()
        if CONSIDER_ANOMALIES.get(r_key, True):
            matched_active_rule = True
            break
            
    if matched_active_rule and EXCLUDE_ANOMALIES_FROM_TRAINING:
        final_excluded_anomalies.append(anom)
    else:
        # Re-admit into clean training annotations if rule was disabled or mode is Audit Only
        final_retained_annotations.append(anom.get('annotation', anom))

# Add base clean annotations
final_retained_annotations.extend(clean_annotations)

retained_pct = (len(final_retained_annotations) / max(1, total_boxes)) * 100
excluded_pct = (len(final_excluded_anomalies) / max(1, total_boxes)) * 100
print(f'Final Training Box Count: {len(final_retained_annotations)} / {total_boxes} ({retained_pct:.1f}% retained)')
print(f'Excluded Anomaly Boxes:   {len(final_excluded_anomalies)} ({excluded_pct:.1f}% excluded)')

# Export the active training dataset for Step 03
final_training_coco = {
    'info': {
        'description': f'RF-DETR Multi-Class Training Dataset ({"Cleaned" if EXCLUDE_ANOMALIES_FROM_TRAINING else "Full Audit"})',
        'exclude_anomalies': EXCLUDE_ANOMALIES_FROM_TRAINING,
        'active_rules': CONSIDER_ANOMALIES
    },
    'images': list({
        a.get('image_id', idx): {
            'id': a.get('image_id', idx),
            'file_name': Path(str(a.get('image_url', f'{idx}.jpg'))).name,
            'coco_url': a.get('image_url')
        } for idx, a in enumerate(final_retained_annotations)
    }.values()),
    'annotations': final_retained_annotations,
    'categories': categories_list
}

training_export_path = CLEANED_DATASET_DIR / '_annotations.coco.json'
with open(training_export_path, 'w', encoding='utf-8') as f:
    json.dump(final_training_coco, f, indent=2)

# Also mirror to REPO_ROOT / cleaned_dataset so Step 03 finds it regardless of current working directory
mirror_export_path = REPO_ROOT / 'cleaned_dataset' / '_annotations.coco.json'
mirror_export_path.parent.mkdir(parents=True, exist_ok=True)
with open(mirror_export_path, 'w', encoding='utf-8') as f:
    json.dump(final_training_coco, f, indent=2)

# Save updated manifest
manifest_export_path = OUTPUT_ANOMALIES_DIR / 'anomalies_manifest.json'
with open(manifest_export_path, 'w', encoding='utf-8') as f:
    json.dump({
        'exclude_anomalies_from_training': EXCLUDE_ANOMALIES_FROM_TRAINING,
        'consider_anomalies_rules': CONSIDER_ANOMALIES,
        'total_audited': total_boxes,
        'clean_retained_count': len(final_retained_annotations),
        'excluded_anomalies_count': len(final_excluded_anomalies),
        'excluded_anomalies': final_excluded_anomalies
    }, f, indent=2)

print()
print(f'Saved training dataset: {training_export_path}')
print(f'Saved updated manifest: {manifest_export_path}')
print('=' * 80)
print('PROCEED TO STEP 03: Run 03_stratified_train_val_test_split.ipynb to generate stratified splits.')
print('=' * 80)
